# Chapitre 18 - Exercices 18.11 à 18.15

Ce notebook traite cinq exercices du chapitre 18 (*problèmes en grande dimension,
$p \gg N$*) :

- **18.11** : la régression logistique n'a pas d'estimateur du maximum de vraisemblance
  quand les classes sont séparables ;
- **18.12** : validation croisée de la ridge avec la réduction $X = RV^T$ calculée une
  seule fois ;
- **18.13** : régression logistique ridge à partir de la seule matrice des produits
  scalaires $K = XX^T$ (et d'un noyau quelconque) ;
- **18.14** : le 1-plus-proche-voisin pondéré par les distances, vu comme une
  analyse discriminante non paramétrique ;
- **18.15** : ACP à noyau et projection d'un nouveau point.

Les fonctions utilisées sont dans le module `esl/ch18_grande_dimension_complements.py`.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch18_grande_dimension_complements as g18
set_style()

## Exercice 18.11 : Classes séparables : la logistique n'a pas de maximum

**Énoncé (reformulé).** On dispose de $N$ couples $(x_i, y_i)$ avec $x_i \in \mathbb{R}$ et
$y_i \in \{0, 1\}$. On suppose les deux classes **séparées** : il existe $C > 0$ tel que,
pour tout $i$ de la classe $0$ et tout $i'$ de la classe $1$, $x_{i'} - x_i \ge C$.
On veut ajuster par maximum de vraisemblance le modèle logistique
$\operatorname{logit} \Pr(Y = 1 \mid X) = \alpha + \beta X$.
Montrer que $\hat\beta$ n'est pas défini.

### Rappels

On note $\sigma(\eta) = \dfrac{1}{1 + e^{-\eta}}$ la fonction logistique et
$\eta_i = \alpha + \beta x_i$ le prédicteur linéaire. Le modèle dit
$\Pr(Y = 1 \mid x_i) = \sigma(\eta_i)$. La log-vraisemblance s'écrit

$$
\ell(\alpha, \beta) = \sum_{i=1}^N \Big[ y_i \log \sigma(\eta_i)
  + (1 - y_i) \log\big(1 - \sigma(\eta_i)\big) \Big].
$$

Chaque terme est le logarithme d'une probabilité strictement comprise entre $0$ et $1$
(car $0 < \sigma(\eta) < 1$ pour tout réel $\eta$) : **chaque terme est strictement
négatif**, donc $\ell(\alpha, \beta) < 0$ pour tout $(\alpha, \beta) \in \mathbb{R}^2$.

### Étape 1 : la borne supérieure de $\ell$ vaut $0$

Notons $a = \max\{x_i : y_i = 0\}$ et $b = \min\{x_{i'} : y_{i'} = 1\}$. L'hypothèse de
séparation donne $b - a \ge C > 0$. Posons $c = (a + b)/2$, le milieu de « l'espace vide »
entre les deux classes. Alors

- si $y_i = 1$ : $x_i - c \ge b - c = (b - a)/2 \ge C/2$ ;
- si $y_i = 0$ : $x_i - c \le a - c = -(b - a)/2 \le -C/2$.

Choisissons $\beta = n$ et $\alpha = -nc$, de sorte que $\eta_i = n(x_i - c)$.
Quand $n \to +\infty$ :

- pour $y_i = 1$, $\eta_i \ge nC/2 \to +\infty$ donc $\log \sigma(\eta_i) \to 0$ ;
- pour $y_i = 0$, $\eta_i \le -nC/2 \to -\infty$ donc $\log(1 - \sigma(\eta_i)) \to 0$.

Par somme d'un nombre fini de termes, $\ell(-nc, n) \to 0$. Comme $\ell < 0$ partout,
on conclut que $\sup_{(\alpha, \beta)} \ell(\alpha, \beta) = 0$.

### Étape 2 : cette borne n'est jamais atteinte

Le maximum de vraisemblance serait un point $(\hat\alpha, \hat\beta)$ tel que
$\ell(\hat\alpha, \hat\beta) = \sup \ell = 0$. Or $\ell(\alpha, \beta) < 0$ en **tout**
point de $\mathbb{R}^2$ : un tel point n'existe pas. L'estimateur du maximum de
vraisemblance n'est donc pas défini.

On peut préciser ce que font les suites « maximisantes » $(\alpha_n, \beta_n)$ avec
$\ell(\alpha_n, \beta_n) \to 0$. Chaque terme (négatif) doit tendre vers $0$, donc
$\eta_{i'} \to +\infty$ pour toute observation $i'$ de la classe $1$ et
$\eta_i \to -\infty$ pour toute observation $i$ de la classe $0$. En particulier

$$
\beta_n (x_{i'} - x_i) = \eta_{i'} - \eta_i \longrightarrow +\infty ,
$$

et comme $0 < C \le x_{i'} - x_i$ est fixé, nécessairement $\beta_n \to +\infty$ :
« $\hat\beta = +\infty$ ». C'est ce qu'on entend par *$\hat\beta$ non défini*.

> **Intuition.** Quand les classes sont séparées, plus la pente est raide, plus la
> sigmoïde ressemble à une marche placée dans le trou entre les classes, et plus chaque
> probabilité ajustée se rapproche de $0$ ou de $1$ : la vraisemblance s'améliore sans fin.

### Vérification numérique

On simule deux classes séparées sur la droite réelle et on lance une montée de gradient
sur $\ell$ (fonction `logistic_gradient_ascent`). Si un maximum existait, les
coefficients se stabiliseraient.

In [2]:
rng = np.random.RandomState(0)
x0 = rng.uniform(-3, -0.5, 15)
x1 = rng.uniform(0.5, 3, 15)
x = np.r_[x0, x1][:, None]
y = np.r_[np.zeros(15), np.ones(15)]
print("écart minimal entre classes C =", round(x1.min() - x0.max(), 3))

écart minimal entre classes C = 1.141


In [ ]:
iters, path, ll = g18.logistic_gradient_ascent(x, y, n_iter=30000, lr=2.0)
print("itération   beta      alpha    log-vraisemblance")
for k in np.linspace(0, len(iters) - 1, 6).astype(int):
    print("%9d  %8.3f  %8.3f  %.2e" % (iters[k], path[k, 1], path[k, 0], ll[k]))

La pente $\beta$ croît sans s'arrêter (de façon logarithmique en le nombre d'itérations)
tandis que la log-vraisemblance tend vers $0$ par valeurs négatives.